# Milestone 3: Artist Similarity Search
Kevin — CSC 475, Music Maven (Group 4)

Objective 1, PI.1–PI.4 (using synthetic data to be replace by Liam's Milestone 2 results):
- PI.1: Weighted Euclidean distance between artist profiles
- PI.2: Ranked top-k similar artists
- PI.3: Euclidean vs cosine comparison across feature subsets
- PI.4: Filtering by genre, year range, and metadata constraints

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Generation
Fake artist profiles for development purposes, each artist gets MIR features (tempo, energy, valence, danceability, key, mode). Real Music4All data will replace it later after Liam's milestone 2 is completed

In [ ]:
# genre-specific feature distributions: (mean, std) for each MIR feature
GENRE_PROFILES = {
    "rock": {
        "tempo": (120.0, 15.0), "energy": (0.70, 0.12), "valence": (0.50, 0.15),
        "danceability": (0.50, 0.13), "key": (5.5, 3.5), "mode": (0.60, 0.20),
    },
    "electronic": {
        "tempo": (128.0, 12.0), "energy": (0.80, 0.10), "valence": (0.55, 0.15),
        "danceability": (0.75, 0.12), "key": (5.5, 3.5), "mode": (0.50, 0.22),
    },
    "jazz": {
        "tempo": (110.0, 18.0), "energy": (0.40, 0.13), "valence": (0.45, 0.14),
        "danceability": (0.40, 0.13), "key": (5.5, 3.5), "mode": (0.50, 0.22),
    },
    "pop": {
        "tempo": (115.0, 14.0), "energy": (0.65, 0.12), "valence": (0.65, 0.14),
        "danceability": (0.70, 0.12), "key": (5.5, 3.5), "mode": (0.70, 0.18),
    },
    "hip-hop": {
        "tempo": (95.0, 13.0), "energy": (0.70, 0.12), "valence": (0.50, 0.15),
        "danceability": (0.80, 0.10), "key": (5.5, 3.5), "mode": (0.45, 0.22),
    },
    "classical": {
        "tempo": (90.0, 20.0), "energy": (0.25, 0.12), "valence": (0.35, 0.13),
        "danceability": (0.20, 0.10), "key": (5.5, 3.5), "mode": (0.55, 0.20),
    },
}

# year ranges per genre: (earliest_start, latest_start, earliest_end, latest_end)
_YEAR_RANGES = {
    "rock": (1960, 1990, 1975, 2020), "electronic": (1980, 2010, 1995, 2025),
    "jazz": (1940, 1980, 1960, 2020), "pop": (1960, 2010, 1975, 2025),
    "hip-hop": (1980, 2010, 1990, 2025), "classical": (1700, 1970, 1900, 2020),
}

# tag pools for each genre
_GENRE_TAGS = {
    "rock": ["energetic", "guitar-driven", "loud", "anthemic", "raw", "distorted", "live-feel"],
    "electronic": ["synthetic", "groovy", "danceable", "upbeat", "pulsing", "bassy", "futuristic"],
    "jazz": ["chill", "improvisational", "sophisticated", "mellow", "swingy", "smooth", "complex"],
    "pop": ["catchy", "upbeat", "polished", "melodic", "danceable", "radio-friendly", "feel-good"],
    "hip-hop": ["groovy", "lyrical", "bassy", "rhythmic", "urban", "soulful", "punchy"],
    "classical": ["orchestral", "dramatic", "expressive", "refined", "acoustic", "intricate", "majestic"],
}

_GENRES = list(GENRE_PROFILES.keys())

def generate_mock_profiles(n_artists=300, seed=42):
    '''Generate synthetic artist profiles with genre-specific feature distributions.'''
    rng = np.random.default_rng(seed)

    # round-robin genre assignment, then shuffle
    primary_genres = np.array((_GENRES * ((n_artists // len(_GENRES)) + 1))[:n_artists])
    rng.shuffle(primary_genres)

    rows = []
    for idx in range(n_artists):
        genre = primary_genres[idx]
        params = GENRE_PROFILES[genre]

        # sample MIR features from genre distributions
        tempo = float(np.clip(rng.normal(*params["tempo"]), 40.0, 220.0))
        energy = float(np.clip(rng.normal(*params["energy"]), 0.0, 1.0))
        valence = float(np.clip(rng.normal(*params["valence"]), 0.0, 1.0))
        danceability = float(np.clip(rng.normal(*params["danceability"]), 0.0, 1.0))
        mode = float(np.clip(rng.normal(*params["mode"]), 0.0, 1.0))
        key = int(np.clip(round(rng.normal(*params["key"])), 0, 11))

        # year range and tags
        yr_min_lo, yr_min_hi, yr_max_lo, yr_max_hi = _YEAR_RANGES[genre]
        year_min = int(rng.integers(yr_min_lo, yr_min_hi + 1))
        year_max = int(rng.integers(max(year_min + 5, yr_max_lo), yr_max_hi + 1))
        tag_pool = _GENRE_TAGS[genre]
        n_tags = int(rng.integers(2, 5))
        tags_str = ",".join(rng.choice(tag_pool, size=min(n_tags, len(tag_pool)), replace=False).tolist())
        song_count = int(rng.integers(5, 81))

        rows.append({
            "artist_id": f"A{idx+1:03d}", "artist_name": f"Artist_{idx+1:03d}",
            "tempo": tempo, "energy": energy, "valence": valence,
            "danceability": danceability, "key": key, "mode": mode,
            "genres": genre, "tags": tags_str,
            "year_min": year_min, "year_max": year_max, "song_count": song_count,
        })

    return pd.DataFrame(rows)

profiles = generate_mock_profiles()
print(f"{len(profiles)} artist profiles generated")
profiles.head(10)

## PI.1: Weighted Euclidean Distance
Normalize features to a common scale, then compute weighted Euclidean distance:

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{N} w_i (x_i - y_i)^2}$$

Also implementing cosine similarity for the comparison in PI.3:

$$\text{sim}(\mathbf{x}, \mathbf{y}) = \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|}$$

In [ ]:
# features we care about
MIR_FEATURES = ["tempo", "energy", "valence", "danceability", "key", "mode"]

# scale features to [0,1] (minmax) or mean=0/std=1 (zscore), returns (normalized_matrix, params_dict) so we can reuse on queries later
def normalize_features(X, method="minmax"):
    X = np.array(X, dtype=float)

    if method == "minmax":
        mins = X.min(axis=0)
        maxs = X.max(axis=0)
        ranges = maxs - mins
        with np.errstate(invalid="ignore", divide="ignore"):
            X_norm = np.where(ranges == 0, 0.0, (X - mins) / ranges)
        params = {"mins": mins, "maxs": maxs}

    elif method == "zscore":
        means = X.mean(axis=0)
        stds = X.std(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            X_norm = np.where(stds == 0, 0.0, (X - means) / stds)
        params = {"means": means, "stds": stds}

    else:
        raise ValueError(f"Unknown method '{method}'. Use 'minmax' or 'zscore'.")

    return X_norm, params

# apply previously fitted normalization params to a new vector
def apply_normalization(x, params, method="minmax"):
    x = np.array(x, dtype=float)

    if method == "minmax":
        ranges = params["maxs"] - params["mins"]
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(ranges == 0, 0.0, (x - params["mins"]) / ranges)

    elif method == "zscore":
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(params["stds"] == 0, 0.0, (x - params["means"]) / params["stds"])

    else:
        raise ValueError(f"Unknown method '{method}'. Use 'minmax' or 'zscore'.")

# normalize weights to sum to 1, or return uniform if None
def prepare_weights(weights, n_features):
    if weights is None:
        return np.full(n_features, 1.0 / n_features)
    w = np.array(weights, dtype=float)
    total = w.sum()
    if total == 0:
        return np.full(n_features, 1.0 / n_features)
    return w / total

# weighted euclidean distance from query to all candidates (vectorized)
def weighted_euclidean_batch(query, candidates, weights=None):
    query = np.array(query, dtype=float)
    candidates = np.array(candidates, dtype=float)
    w = prepare_weights(weights, query.shape[0])
    diff = candidates - query
    return np.sqrt((w * diff * diff).sum(axis=1))

# cosine similarity from query to all candidates (vectorized)
def cosine_similarity_batch(query, candidates):
    query = np.array(query, dtype=float)
    candidates = np.array(candidates, dtype=float)

    norm_q = np.linalg.norm(query)
    if norm_q == 0.0:
        return np.zeros(candidates.shape[0])

    norms_c = np.linalg.norm(candidates, axis=1)
    dots = candidates @ query
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(norms_c == 0.0, 0.0, dots / (norm_q * norms_c))

# weighted cosine distance (1 - weighted cosine sim) using the sqrt(w) scaling trick
def weighted_cosine_distance_batch(query, candidates, weights=None):
    query = np.array(query, dtype=float)
    candidates = np.array(candidates, dtype=float)
    w = prepare_weights(weights, query.shape[0])
    sqrt_w = np.sqrt(w)
    # scale both query and candidates by sqrt(w), then use regular cosine
    return 1.0 - cosine_similarity_batch(query * sqrt_w, candidates * sqrt_w)

## PI.2 and PI.4: Ranked Top-k Similar Artists and Metadata Filtering
The SimilarityEngine class normalizes all artist profiles once at init, then answers queries by computing batch distances and returning the k closest artists as a ranked DataFrame. It also allows you to filter the candidate pool before computing distances.

In [ ]:
class SimilarityEngine:
    # similarity search over artist profiles

    def __init__(self, profiles_df, normalize="minmax"):
        self._df = profiles_df.reset_index(drop=True)
        self._method = normalize
        raw = self._df[MIR_FEATURES].to_numpy(dtype=float)
        self._norm_matrix, self._norm_params = normalize_features(raw, method=normalize)
        
    # find top-k similar artists, returns dataframe with artist_id, artist_name, distance, rank
    def search(self, artist_id, k=10, features=None, weights=None, metric="euclidean", exclude_genres=None, exclude_year_range=None, require_genres=None):

        # find the query artist
        mask = self._df["artist_id"] == artist_id
        if not mask.any():
            raise ValueError(f"Artist '{artist_id}' not found.")
        query_idx = int(mask.idxmax())

        # filter candidates (removes query artist + applies metadata filters)
        candidates = self.apply_filters(self._df, artist_id, exclude_genres, exclude_year_range, require_genres)

        # figure out which features to use
        feats = features if features is not None else MIR_FEATURES
        feat_idxs = [MIR_FEATURES.index(f) for f in feats]

        # build weight vector aligned to chosen features
        weight_vec = None
        if weights is not None:
            weight_vec = np.array([weights.get(f, 0.0) for f in feats], dtype=float)

        # grab normalized vectors for query and candidates
        query_vec = self._norm_matrix[query_idx, feat_idxs]
        cand_idxs = candidates.index.tolist()
        cand_matrix = self._norm_matrix[np.ix_(cand_idxs, feat_idxs)]

        # compute distances
        if metric == "euclidean":
            dists = weighted_euclidean_batch(query_vec, cand_matrix, weight_vec)
        elif metric == "cosine":
            dists = weighted_cosine_distance_batch(query_vec, cand_matrix, weight_vec)
        else:
            raise ValueError(f"Unknown metric '{metric}'. Use 'euclidean' or 'cosine'.")

        # sort and take top-k
        k = min(k, len(dists))
        order = np.argsort(dists)[:k]

        result = candidates.iloc[order][["artist_id", "artist_name"]].copy()
        result = result.reset_index(drop=True)
        result["distance"] = dists[order]
        result["rank"] = np.arange(1, k + 1)
        return result
    
    # remove query artist and apply genre/year filters
    def apply_filters(self, df, artist_id, exclude_genres, exclude_year_range, require_genres):
        result = df[df["artist_id"] != artist_id].copy()

        # exclude genres
        if exclude_genres:
            excl = {g.lower() for g in exclude_genres}
            def has_excluded(g):
                return bool({x.strip().lower() for x in str(g).split(",")} & excl)
            result = result[~result["genres"].apply(has_excluded)]

        # exclude year range (overlap check)
        if exclude_year_range is not None:
            s, e = exclude_year_range
            overlaps = (result["year_min"] <= e) & (result["year_max"] >= s)
            result = result[~overlaps]

        # require genres
        if require_genres:
            req = {g.lower() for g in require_genres}
            def has_required(g):
                return bool({x.strip().lower() for x in str(g).split(",")} & req)
            result = result[result["genres"].apply(has_required)]

        return result

In [ ]:
engine = SimilarityEngine(profiles)

# example usage: top 10 artists similar to A001 using all features
results = engine.search("A001", k=10)
query_name = profiles.loc[profiles["artist_id"] == "A001", "artist_name"].iloc[0]
query_genre = profiles.loc[profiles["artist_id"] == "A001", "genres"].iloc[0]
print(f"Top 10 similar to {query_name} (genre: {query_genre}):\n")
results

In [ ]:
# examples of applying filters and weights
print("Excluding jazz and classical")
results_filtered = engine.search("A001", k=10, exclude_genres=["jazz", "classical"])
display(results_filtered)

print("\nExcluding 1940-1970, requiring rock")
results_year = engine.search("A001", k=10, exclude_year_range=(1940, 1970), require_genres=["rock"])
display(results_year)

print("\nWeighted: energy=0.4, danceability=0.4, tempo=0.1, valence=0.1")
results_weighted = engine.search("A001", k=10,
    weights={"energy": 0.4, "danceability": 0.4, "tempo": 0.1, "valence": 0.1})
display(results_weighted)

## PI.3: Euclidean vs Cosine Comparison
For each feature subset, run both metrics on several query artists and measure
how much the rankings agree:
- Overlap: fraction of top-k artist IDs appearing in both result sets
- Rank correlation: Spearman correlation of shared artists' ranking positions

In [ ]:
from scipy.stats import spearmanr

# run euclidean and cosine for each (artist, subset) and measure agreement
def compare_metrics(engine, query_artists, feature_subsets, k=10):
    records = []

    for artist_id in query_artists:
        for subset_name, feat_list in feature_subsets.items():
            try:
                res_euc = engine.search(artist_id, k=k, features=feat_list, metric="euclidean")
                res_cos = engine.search(artist_id, k=k, features=feat_list, metric="cosine")
            except ValueError:
                continue

            ids_euc = set(res_euc["artist_id"])
            ids_cos = set(res_cos["artist_id"])
            shared = ids_euc & ids_cos

            # overlap: what fraction of top-k is shared?
            overlap = len(shared) / k if k > 0 else 0.0

            # rank correlation: how similarly are the shared artists ranked?
            rank_corr = float("nan")
            if len(shared) >= 2:
                euc_ranks = []
                cos_ranks = []
                for aid in shared:
                    euc_ranks.append(int(res_euc.loc[res_euc["artist_id"] == aid, "rank"].iloc[0]))
                    cos_ranks.append(int(res_cos.loc[res_cos["artist_id"] == aid, "rank"].iloc[0]))
                rank_corr = float(spearmanr(euc_ranks, cos_ranks).statistic)

            records.append({
                "query_artist": artist_id,
                "feature_subset": subset_name,
                "overlap": overlap,
                "rank_correlation": rank_corr,
                "euclidean_top1": res_euc["artist_id"].iloc[0] if len(res_euc) > 0 else "",
                "cosine_top1": res_cos["artist_id"].iloc[0] if len(res_cos) > 0 else "",
            })

    return pd.DataFrame(records)

In [ ]:
# pick 5 query artists spread across the dataset
n = len(profiles)
sample_ids = [profiles["artist_id"].iloc[i] for i in np.linspace(0, n - 1, 5, dtype=int)]

feature_subsets = {
    "all":    MIR_FEATURES,
    "rhythm": ["tempo", "danceability"],
    "mood":   ["energy", "valence"],
    "tonal":  ["key", "mode"],
}

comparison_df = compare_metrics(engine, sample_ids, feature_subsets, k=10)
comparison_df